# Time-Series Analysis of Indian Port Activity and Economic Output

## Ports Analyzed: Mumbai (JNPT) & Paradip

**Methodology:** Satellite-based ship detection using YOLO11x-OBB with SAHI-style tiling on Sentinel-2 imagery (2018–2026).  
**Data Source:** Microsoft Planetary Computer — Sentinel-2 L2A (monthly, least-cloudy composite)  
**Detection Model:** YOLO11x-OBB with 3x upscaling and 40% tile overlap for small vessel detection  

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.figsize'] = (12, 5)

# Load data
df = pd.read_csv('../ship_counts.csv')
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
df['quarter'] = df['date'].dt.quarter
df['month_name'] = df['date'].dt.strftime('%b')

mumbai = df[df['port'] == 'mumbai'].copy()
paradip = df[df['port'] == 'paradip'].copy()

print(f"Total observations: {len(df)}")
print(f"Mumbai: {len(mumbai)} months | Paradip: {len(paradip)} months")
print(f"Date range: {df['date'].min().strftime('%Y-%m')} to {df['date'].max().strftime('%Y-%m')}")
df.head()

---
## 1. Total Ships Detected Per Year

In [ ]:
# Yearly totals by port
yearly = df.groupby(['port', 'year'])['ship_count'].agg(['sum', 'mean', 'count']).reset_index()
yearly.columns = ['port', 'year', 'total_ships', 'avg_monthly', 'months_observed']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mumbai
m_yearly = yearly[yearly['port'] == 'mumbai']
bars1 = axes[0].bar(m_yearly['year'], m_yearly['total_ships'], color='#2196F3', edgecolor='white', width=0.7)
axes[0].set_title('Mumbai Port — Total Ships Detected Per Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Ships Detected')
for bar, val in zip(bars1, m_yearly['total_ships']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                 str(int(val)), ha='center', va='bottom', fontsize=9, fontweight='bold')

# Paradip
p_yearly = yearly[yearly['port'] == 'paradip']
bars2 = axes[1].bar(p_yearly['year'], p_yearly['total_ships'], color='#FF9800', edgecolor='white', width=0.7)
axes[1].set_title('Paradip Port — Total Ships Detected Per Year', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Total Ships Detected')
for bar, val in zip(bars2, p_yearly['total_ships']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(int(val)), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_yearly_totals.png', bbox_inches='tight')
plt.show()

# Print table
print("\n=== Yearly Ship Count Summary ===")
pivot = yearly.pivot(index='year', columns='port', values='total_ships').fillna(0)
pivot.columns = [c.title() for c in pivot.columns]
print(pivot.to_string())

---
## 2. Month-Wise Ship Counts (Within Year)

In [ ]:
# Select 4 representative years for detailed monthly view
sample_years = [2019, 2020, 2022, 2024]

fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=False)
axes = axes.flatten()
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for idx, year in enumerate(sample_years):
    ax = axes[idx]
    
    m_data = mumbai[mumbai['year'] == year].set_index('month')['ship_count']
    p_data = paradip[paradip['year'] == year].set_index('month')['ship_count']
    
    x = np.arange(1, 13)
    width = 0.35
    
    m_vals = [m_data.get(m, 0) for m in x]
    p_vals = [p_data.get(m, 0) for m in x]
    
    bars_m = ax.bar(x - width/2, m_vals, width, label='Mumbai', color='#2196F3', edgecolor='white')
    bars_p = ax.bar(x + width/2, p_vals, width, label='Paradip', color='#FF9800', edgecolor='white')
    
    ax.set_title(f'{year} — Monthly Ship Counts', fontsize=13, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Ships Detected')
    ax.set_xticks(x)
    ax.set_xticklabels(month_labels, rotation=45)
    ax.legend(loc='upper right')
    ax.set_ylim(bottom=0)

plt.suptitle('Month-Wise Ship Detection — 4 Selected Years', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_monthly_4years.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: all months × all years for Mumbai
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, port, cmap, title in [
    (axes[0], mumbai, 'Blues', 'Mumbai'),
    (axes[1], paradip, 'Oranges', 'Paradip')
]:
    pivot = port.pivot_table(index='month', columns='year', values='ship_count', aggfunc='sum')
    pivot.index = [month_labels[m-1] for m in pivot.index]
    
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap=cmap, ax=ax, 
                linewidths=0.5, cbar_kws={'label': 'Ships'})
    ax.set_title(f'{title} — Ship Count Heatmap (Month × Year)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Month')
    ax.set_xlabel('Year')

plt.tight_layout()
plt.savefig('fig_heatmap.png', bbox_inches='tight')
plt.show()

---
## 3. Additional Analysis

### 3a. Time-Series Trend with Moving Average

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

for port_name, port_df, color in [('Mumbai', mumbai, '#2196F3'), ('Paradip', paradip, '#FF9800')]:
    port_sorted = port_df.sort_values('date')
    ax.plot(port_sorted['date'], port_sorted['ship_count'], 
            alpha=0.3, color=color, linewidth=1)
    # 6-month rolling average
    rolling = port_sorted.set_index('date')['ship_count'].rolling(window=6, min_periods=3).mean()
    ax.plot(rolling.index, rolling.values, 
            color=color, linewidth=2.5, label=f'{port_name} (6-mo avg)')

ax.set_title('Ship Activity Trend — Mumbai vs Paradip (2018–2026)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Ships Detected per Month')
ax.legend(fontsize=12)
ax.axvspan(pd.Timestamp('2020-03-25'), pd.Timestamp('2020-06-30'), 
           alpha=0.15, color='red', label='COVID-19 Lockdown')
ax.annotate('COVID-19\nLockdown', xy=(pd.Timestamp('2020-05-01'), ax.get_ylim()[1]*0.85),
            fontsize=9, color='red', ha='center', fontstyle='italic')

plt.tight_layout()
plt.savefig('fig_trend_line.png', bbox_inches='tight')
plt.show()

### 3b. Seasonality Analysis

In [ ]:
# Average ships by month across all years (seasonality)
fig, ax = plt.subplots(figsize=(12, 5))

for port_name, port_df, color in [('Mumbai', mumbai, '#2196F3'), ('Paradip', paradip, '#FF9800')]:
    seasonal = port_df.groupby('month')['ship_count'].agg(['mean', 'std']).reset_index()
    ax.plot(seasonal['month'], seasonal['mean'], 'o-', color=color, 
            linewidth=2, markersize=8, label=f'{port_name}')
    ax.fill_between(seasonal['month'], 
                    seasonal['mean'] - seasonal['std'],
                    seasonal['mean'] + seasonal['std'],
                    alpha=0.15, color=color)

ax.set_title('Seasonal Pattern — Average Ships by Month (All Years)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Ships Detected')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('fig_seasonality.png', bbox_inches='tight')
plt.show()

### 3c. Year-over-Year Growth Rate

In [ ]:
# YoY growth
fig, ax = plt.subplots(figsize=(14, 5))

for port_name, color in [('mumbai', '#2196F3'), ('paradip', '#FF9800')]:
    p_yearly = yearly[yearly['port'] == port_name].sort_values('year').copy()
    # Only full years (exclude partial 2026)
    p_yearly = p_yearly[p_yearly['months_observed'] >= 10]
    p_yearly['yoy_growth'] = p_yearly['total_ships'].pct_change() * 100
    p_valid = p_yearly.dropna(subset=['yoy_growth'])
    
    bars = ax.bar(p_valid['year'] + (0.2 if port_name == 'paradip' else -0.2), 
                  p_valid['yoy_growth'], width=0.35,
                  color=color, label=port_name.title(), edgecolor='white')

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-')
ax.set_title('Year-over-Year Growth in Ship Activity (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('YoY Growth (%)')
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('fig_yoy_growth.png', bbox_inches='tight')
plt.show()

### 3d. COVID-19 Impact Analysis

In [ ]:
# Pre-COVID (2018-2019) vs COVID (2020-2021) vs Post-COVID (2022-2024)
def categorize_period(year):
    if year <= 2019:
        return 'Pre-COVID (2018-19)'
    elif year <= 2021:
        return 'COVID (2020-21)'
    else:
        return 'Post-COVID (2022+)'

df['period'] = df['year'].apply(categorize_period)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

period_order = ['Pre-COVID (2018-19)', 'COVID (2020-21)', 'Post-COVID (2022+)']
colors = ['#4CAF50', '#F44336', '#2196F3']

for ax, port_name in [(axes[0], 'mumbai'), (axes[1], 'paradip')]:
    port_data = df[df['port'] == port_name]
    period_avg = port_data.groupby('period')['ship_count'].mean().reindex(period_order)
    
    bars = ax.bar(period_order, period_avg, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, period_avg):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_title(f'{port_name.title()} — Avg Monthly Ships by Period', fontsize=13, fontweight='bold')
    ax.set_ylabel('Avg Ships per Month')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('fig_covid_impact.png', bbox_inches='tight')
plt.show()

---
## 4. Mumbai vs Paradip — Year-Wise Comparison

In [ ]:
# Side-by-side yearly comparison
fig, ax = plt.subplots(figsize=(14, 6))

m_yearly = yearly[yearly['port'] == 'mumbai'].sort_values('year')
p_yearly = yearly[yearly['port'] == 'paradip'].sort_values('year')

# Only common years
common_years = sorted(set(m_yearly['year']) & set(p_yearly['year']))
m_vals = m_yearly[m_yearly['year'].isin(common_years)].set_index('year')['total_ships']
p_vals = p_yearly[p_yearly['year'].isin(common_years)].set_index('year')['total_ships']

x = np.arange(len(common_years))
width = 0.35

bars1 = ax.bar(x - width/2, [m_vals.get(y, 0) for y in common_years], 
               width, label='Mumbai', color='#2196F3', edgecolor='white')
bars2 = ax.bar(x + width/2, [p_vals.get(y, 0) for y in common_years],
               width, label='Paradip', color='#FF9800', edgecolor='white')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2, height + 1,
                    str(int(height)), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Mumbai vs Paradip — Yearly Ship Count Comparison', fontsize=15, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Total Ships Detected', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(common_years)
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('fig_port_comparison_yearly.png', bbox_inches='tight')
plt.show()

In [ ]:
# Ratio analysis: Mumbai / Paradip over time
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Yearly ratio
ratio_df = pd.DataFrame({'year': common_years})
ratio_df['mumbai'] = [m_vals.get(y, 0) for y in common_years]
ratio_df['paradip'] = [p_vals.get(y, 0) for y in common_years]
ratio_df['ratio'] = ratio_df['mumbai'] / ratio_df['paradip'].replace(0, np.nan)

axes[0].plot(ratio_df['year'], ratio_df['ratio'], 'o-', color='#9C27B0', linewidth=2, markersize=8)
axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Mumbai-to-Paradip Ship Ratio (Year-wise)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Ratio (Mumbai / Paradip)')
axes[0].annotate('Ratio > 1 = Mumbai busier', xy=(0.5, 0.02), xycoords='axes fraction',
                 fontsize=9, color='gray', ha='center', fontstyle='italic')

# Cumulative comparison
m_sorted = mumbai.sort_values('date').copy()
p_sorted = paradip.sort_values('date').copy()
m_sorted['cumulative'] = m_sorted['ship_count'].cumsum()
p_sorted['cumulative'] = p_sorted['ship_count'].cumsum()

axes[1].plot(m_sorted['date'], m_sorted['cumulative'], color='#2196F3', linewidth=2, label='Mumbai')
axes[1].plot(p_sorted['date'], p_sorted['cumulative'], color='#FF9800', linewidth=2, label='Paradip')
axes[1].set_title('Cumulative Ship Detections Over Time', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cumulative Ships')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig_ratio_cumulative.png', bbox_inches='tight')
plt.show()

---
## 5. Statistical Summary & Key Findings

In [ ]:
print("=" * 70)
print("  STATISTICAL SUMMARY")
print("=" * 70)

for port_name, port_df in [('Mumbai', mumbai), ('Paradip', paradip)]:
    print(f"\n--- {port_name} ---")
    print(f"  Total observations:    {len(port_df)} months")
    print(f"  Total ships detected:  {port_df['ship_count'].sum():.0f}")
    print(f"  Mean ships/month:      {port_df['ship_count'].mean():.1f}")
    print(f"  Median ships/month:    {port_df['ship_count'].median():.1f}")
    print(f"  Std deviation:         {port_df['ship_count'].std():.1f}")
    print(f"  Min:                   {port_df['ship_count'].min():.0f} ({port_df.loc[port_df['ship_count'].idxmin(), 'date'].strftime('%Y-%m')})")
    print(f"  Max:                   {port_df['ship_count'].max():.0f} ({port_df.loc[port_df['ship_count'].idxmax(), 'date'].strftime('%Y-%m')})")
    
    # Trend (linear regression)
    port_sorted = port_df.sort_values('date')
    x_numeric = np.arange(len(port_sorted))
    slope, intercept, r_value, p_value, std_err = stats.linregress(x_numeric, port_sorted['ship_count'])
    trend_dir = 'increasing' if slope > 0 else 'decreasing'
    print(f"  Overall trend:         {trend_dir} ({slope:+.3f} ships/month, R²={r_value**2:.3f}, p={p_value:.4f})")

# Correlation between ports
merged = mumbai[['date', 'ship_count']].merge(
    paradip[['date', 'ship_count']], on='date', suffixes=('_mumbai', '_paradip')
)
if len(merged) > 5:
    corr, p = stats.pearsonr(merged['ship_count_mumbai'], merged['ship_count_paradip'])
    print(f"\n--- Port Correlation ---")
    print(f"  Pearson r = {corr:.3f} (p = {p:.4f})")
    print(f"  Interpretation: {'Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.4 else 'Weak'} {'positive' if corr > 0 else 'negative'} correlation")

In [ ]:
# Scatter plot: Mumbai vs Paradip monthly ships
if len(merged) > 5:
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(merged['ship_count_mumbai'], merged['ship_count_paradip'], 
               alpha=0.6, s=60, color='#673AB7', edgecolors='white')
    
    # Regression line
    z = np.polyfit(merged['ship_count_mumbai'], merged['ship_count_paradip'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(merged['ship_count_mumbai'].min(), merged['ship_count_mumbai'].max(), 100)
    ax.plot(x_line, p(x_line), '--', color='gray', linewidth=1.5, alpha=0.7)
    
    ax.set_title(f'Mumbai vs Paradip — Monthly Ship Correlation (r={corr:.3f})', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Mumbai Ships/Month', fontsize=12)
    ax.set_ylabel('Paradip Ships/Month', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('fig_port_scatter.png', bbox_inches='tight')
    plt.show()

---
## 6. Quarterly Analysis

In [ ]:
# Quarterly boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, port_name, port_df, color_pal in [
    (axes[0], 'Mumbai', mumbai, 'Blues'),
    (axes[1], 'Paradip', paradip, 'Oranges')
]:
    sns.boxplot(data=port_df, x='quarter', y='ship_count', 
                palette=color_pal, ax=ax, width=0.5)
    ax.set_title(f'{port_name} — Ship Count by Quarter', fontsize=13, fontweight='bold')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Ships Detected')
    ax.set_xticklabels(['Q1\n(Jan-Mar)', 'Q2\n(Apr-Jun)', 'Q3\n(Jul-Sep)', 'Q4\n(Oct-Dec)'])

plt.tight_layout()
plt.savefig('fig_quarterly_box.png', bbox_inches='tight')
plt.show()

---
## 7. Key Findings & Conclusions

The detailed findings and economic interpretation are provided in the accompanying PDF report.

In [ ]:
# Summary table for report
print("\n" + "=" * 70)
print("  YEAR-WISE COMPARISON TABLE")
print("=" * 70)

comparison = yearly.pivot_table(index='year', columns='port', 
                                values=['total_ships', 'avg_monthly', 'months_observed'])
comparison.columns = [f"{col[1].title()}_{col[0]}" for col in comparison.columns]
print(comparison.to_string())
comparison.to_csv('yearly_comparison_table.csv')
print("\nSaved to yearly_comparison_table.csv")